In [1]:
!pip install findspark

# El Optimizador Spark Catalyst

La forma más fácil de escribir aplicaciones de procesamiento de datos eficientes es no tener que preocuparse por la optimización manual. Esta es precisamente la promesa de **Spark Catalyst**, el optimizador de consultas extensible que constituye el segundo componente principal del módulo Spark SQL.

Catalyst desempeña un papel crucial al garantizar que la lógica de procesamiento escrita mediante la API de DataFrames o SQL se ejecute de manera rápida y eficiente. Fue diseñado con dos objetivos principales:
1. Minimizar los tiempos de respuesta de las consultas de un extremo a otro (*end-to-end*).
2. Ser altamente extensible, permitiendo a los usuarios avanzados inyectar código personalizado para realizar optimizaciones específicas.

---

## El Flujo de Optimización a Alto Nivel

A alto nivel, Spark Catalyst traduce la lógica de procesamiento escrita por el usuario a través de tres fases principales hasta convertirla en código ejecutable:



### 1. El Plan Lógico (*Logical Plan*)
El primer paso en el proceso de optimización es crear un **Plan Lógico** a partir del objeto DataFrame o del Árbol de Sintaxis Abstracta (AST) generado al analizar una consulta SQL. Este proceso se divide en dos subetapas:

* **Resolución de Referencias:** Catalyst analiza el plan lógico inicial (no resuelto) contrastándolo con el *Catalog* (metadatos de las tablas y columnas) para validar que los nombres de las columnas y los tipos de datos existan y sean correctos.
* **Optimización del Plan Lógico:** Una vez resuelto, Catalyst aplica un conjunto de optimizaciones basadas en **reglas (RBO)** y basadas en **costos (CBO)**.

> 💡 **Principio de Optimización:**
> Ambos enfoques siguen la regla de oro de **filtrar y podar los datos innecesarios lo antes posible** (como el empuje de predicados o *predicate pushdown* y la proyección de columnas) para minimizar el costo de procesamiento por cada operador.

### 2. El Plan Físico (*Physical Plan*)
Una vez optimizado el plan lógico, Catalyst genera uno o más **Planes Físicos** que describen exactamente cómo se ejecutará la consulta en el clúster balanceando los recursos distribuidos. Utilizando su modelo de costo, selecciona el plan físico más eficiente (por ejemplo, decidiendo si utilizar un *Broadcast Hash Join* o un *Shuffle Hash Join*).

### 3. Generación de Código (*Code Generation*)
El paso final consiste en convertir el plan físico seleccionado en código de bytes (*bytecode*) de Java altamente optimizado para que corra directamente en las JVM de los ejecutores, utilizando una característica llamada *Whole-Stage Code Generation*.

In [2]:
import findspark

In [3]:
findspark.init()

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.appName("Catalyst Optimyzer").getOrCreate()

In [8]:
data = spark.read.parquet("./data/")

In [9]:
# Vamos a analizar un ejmpl donde vamos a leer los datos de vuelo

# luego haremos un filtrado del mes y agregar un columna y proyectar la columna y finalmente filtramos las filas segun la aerolinea

In [10]:
data.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [11]:
data.show()

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

In [12]:
from pyspark.sql.functions import col

In [15]:
nuevo_df = (data.filter(col("MONTH").isin(6,7,8))
           .withColumn("dis_tiempo_aire", col("DISTANCE") / col("AIR_TIME") )
).select(
    col("AIRLINE"),
    col("dis_tiempo_aire")
).where(col("AIRLINE").isin("AA", "DL", "AS"))

In [16]:
# Hemos creado nuestro nuevo df ahora como podemos visualizar el plan logico y el plan fisico

In [17]:
nuevo_df.explain(True)

== Parsed Logical Plan ==
'Filter 'in('AIRLINE, AA, DL, AS)
+- Project [AIRLINE#35, dis_tiempo_aire#157]
   +- Project [YEAR#31, MONTH#32, DAY#33, DAY_OF_WEEK#34, AIRLINE#35, FLIGHT_NUMBER#36, TAIL_NUMBER#37, ORIGIN_AIRPORT#38, DESTINATION_AIRPORT#39, SCHEDULED_DEPARTURE#40, DEPARTURE_TIME#41, DEPARTURE_DELAY#42, TAXI_OUT#43, WHEELS_OFF#44, SCHEDULED_TIME#45, ELAPSED_TIME#46, AIR_TIME#47, DISTANCE#48, WHEELS_ON#49, TAXI_IN#50, SCHEDULED_ARRIVAL#51, ARRIVAL_TIME#52, ARRIVAL_DELAY#53, DIVERTED#54, CANCELLED#55, ... 7 more fields]
      +- Filter MONTH#32 IN (6,7,8)
         +- Relation [YEAR#31,MONTH#32,DAY#33,DAY_OF_WEEK#34,AIRLINE#35,FLIGHT_NUMBER#36,TAIL_NUMBER#37,ORIGIN_AIRPORT#38,DESTINATION_AIRPORT#39,SCHEDULED_DEPARTURE#40,DEPARTURE_TIME#41,DEPARTURE_DELAY#42,TAXI_OUT#43,WHEELS_OFF#44,SCHEDULED_TIME#45,ELAPSED_TIME#46,AIR_TIME#47,DISTANCE#48,WHEELS_ON#49,TAXI_IN#50,SCHEDULED_ARRIVAL#51,ARRIVAL_TIME#52,ARRIVAL_DELAY#53,DIVERTED#54,CANCELLED#55,... 6 more fields] parquet

== Analyze

## Termino de Podar la consulta para optimizarla lo maximo posible

In [18]:
# lo que nos muestra es el plan logico : lo que hacmeos nuestra base relacional lo que filtramos crear un proyeccio (o crear una nueva columna ) hacer un filtro etc

In [19]:
# En el plan Optimized Logical Plan : ha combinado los dos filtros en un solo paso (el filtro del mes y el filtro de la aerolinea)

In [20]:
# Plan fisico : lee nuestro archiovo parquet -- y luego realiza el filtro -- Hace la operacion de proyeccion y esta un poco mas compacto y mas optimizado